# Test de-identification code for 1000genomesVDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1755858350179_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-119-229.ap-southeast-1.compute.internal:39075
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
# source
trust_prefix = 's3://precise-trust/opendata-1000genomes/'

# input
vds_uri = trust_prefix + '1000genomes-vds-n3205.vds'

# output
hashed_vds_uri = trust_prefix + '1000genomes-vds-n3205.hashed.vds'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# read vds
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
# describe reference_data
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [6]:
# check ref_block_max_length
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

151393

In [7]:
# describe variant_data
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'sample_info': struct {
        npmid: str, 
        sex: str, 
        pid: str, 
        population_code: str, 
        population_name: str, 
        superpopulation_code: str, 
        superpopulation_name: str, 
        data_collections: array<str>
    }
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64

In [8]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 3205
Number of variant partitions: 5767
Total number of variants: 156,228,032

In [9]:
# quick sanity
print("reference_data col key:", vds.reference_data.col_key)
print("variant_data  col key:", vds.variant_data.col_key)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

reference_data col key: <StructExpression of type struct{s: str}>
variant_data  col key: <StructExpression of type struct{s: str}>

In [10]:
# create a vds_subset for testing

# 1) take first 10 sample IDs
keep = [c.s for c in vds.variant_data.cols().select().take(10)]

# 2) filter samples (list[str] is allowed)
vds_s = hl.vds.filter_samples(vds, keep)

# 3) define a tiny interval (adjust contig style 'chr20' vs '20')
iv = hl.parse_locus_interval("chr20:1-5000000", reference_genome="GRCh38")

# 4) filter by interval(s) — use filter_intervals for VDS
vds_subset = hl.vds.filter_intervals(vds_s, [iv])

# quick peek
print("n cols:", vds_subset.variant_data.count_cols())
print("n var rows:", vds_subset.variant_data.count_rows())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

n cols: 10
n var rows: 21386
2025-08-22 11:01:01.461 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'
2025-08-22 11:01:05.218 Hail: INFO: Coerced sorted dataset

In [11]:
# replace original vds by vds_subset
vds_full = vds           # keep original
vds = vds_subset         # overwrite for quick testing

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# Preferred: set an environment variable before launching Jupyter:
#   export DEID_SALT="your-long-random-secret"
# SALT = os.getenv("DEID_SALT")

# For testing only, hardcode a fixed salt here:
SALT = "1234567890abcdef1234567890abcdef1234567890abcdef1234567890abcdef"

print("SALT (for test only):", SALT)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SALT (for test only): 1234567890abcdef1234567890abcdef1234567890abcdef1234567890abcdef

In [13]:
# ----------------------------------------------------------------------
# Build a per-sample token table with SHA-256 (using pandas + hashlib).
# ----------------------------------------------------------------------

import pandas as pd
import hashlib

# 1) collect sample metadata to pandas
cols_df = vds.variant_data.cols().to_pandas()

# inspect what you got
print(cols_df.columns.tolist()[:10])  # should include 's', 'sample_info.npmid', 'sample_info.pid'

# 2) helper for salted SHA256
def sha256_hex(value: str, salt: str) -> str:
    value = "" if value is None else str(value)
    return hashlib.sha256((salt + value).encode("utf-8")).hexdigest()

# 3) compute hashes
cols_df["hash_npmid"] = cols_df["sample_info.npmid"].apply(lambda x: sha256_hex(x, SALT))
cols_df["hash_pid"]   = cols_df["sample_info.pid"].apply(lambda x: sha256_hex(x, SALT))

# 4) keep only what we need
token_df = cols_df[["s", "hash_npmid", "hash_pid"]].copy()

# 5) back to hail, keyed by s
ht_token = hl.Table.from_pandas(token_df, key="s")

# 6) quick peek
ht_token.describe()
ht_token.show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['s', 'sample_info.npmid', 'sample_info.sex', 'sample_info.pid', 'sample_info.population_code', 'sample_info.population_name', 'sample_info.superpopulation_code', 'sample_info.superpopulation_name', 'sample_info.data_collections']
----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    's': str 
    'hash_npmid': str 
    'hash_pid': str 
----------------------------------------
Key: ['s']
----------------------------------------
+-----------+
| s         |
+-----------+
| str       |
+-----------+
| "HG00096" |
| "HG00097" |
| "HG00099" |
| "HG00100" |
| "HG00101" |
+-----------+

+--------------------------------------------------------------------+
| hash_npmid                                                         |
+--------------------------------------------------------------------+
| str                                                                |
+------------------------------------------------------------

In [14]:
# ----------------------------------------------------------------------
# Step 1: Re-key both variant_data and reference_data with hashed IDs
#         and rebuild a consistent de-identified VDS.
# ----------------------------------------------------------------------

# --- variant_data ---
vd = vds.variant_data.annotate_cols(
    # bring in hashed IDs from ht_token (joined on the current column key 's')
    _new_s      = ht_token[vds.variant_data.col_key].hash_npmid,
    sample_info = vds.variant_data.sample_info.annotate(
        # overwrite npmid and pid inside sample_info with their hashed values
        npmid = ht_token[vds.variant_data.col_key].hash_npmid,
        pid   = ht_token[vds.variant_data.col_key].hash_pid,
    ),
)

# now re-key columns: the new 's' is the hashed npmid
# drop the temporary helper column after re-keying
vd = vd.key_cols_by(s = vd._new_s).drop('_new_s')


# --- reference_data ---
rd = vds.reference_data.annotate_cols(
    # only need the new key (hashed npmid); no sample_info here
    _new_s = ht_token[vds.reference_data.col_key].hash_npmid
)

# re-key columns by the hashed ID, then drop the helper
rd = rd.key_cols_by(s = rd._new_s).drop('_new_s')


# --- rebuild the VariantDataset with the two updated MTs ---
vds_deid = hl.vds.VariantDataset(rd, vd)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
# ----------------------------------------------------------------------
# Step 2: Sanity checks to make sure the de-identification worked
# ----------------------------------------------------------------------

vref = vds_deid.reference_data
vvar = vds_deid.variant_data

# 1) keys must match across reference_data and variant_data
ref_keys = vref.cols().select()
var_keys = vvar.cols().select()
assert ref_keys.anti_join(var_keys).count() == 0, "Some keys missing in variant_data"
assert var_keys.anti_join(ref_keys).count() == 0, "Some keys missing in reference_data"

# 2) quick info: column counts should be identical
print("n cols (ref, var):", vref.count_cols(), vvar.count_cols())

# 3) validate hashed IDs
key_ht = vvar.cols()

# a) all keys are valid sha256 hex strings (64 lowercase hex chars)
assert key_ht.aggregate(
    hl.agg.count_where(~key_ht.s.matches(r'^[0-9a-f]{64}$'))
) == 0, "Some column keys are not 64-char sha256 hex"

# b) sample_info.npmid must equal the new key
assert key_ht.aggregate(
    hl.agg.count_where(key_ht.sample_info.npmid != key_ht.s)
) == 0, "sample_info.npmid != column key for some samples"

# c) sample_info.pid must also be sha256 hex
assert key_ht.aggregate(
    hl.agg.count_where(~key_ht.sample_info.pid.matches(r'^[0-9a-f]{64}$'))
) == 0, "Some sample_info.pid are not 64-char sha256 hex"

# 4) guard against missing or duplicate hashed IDs
assert key_ht.aggregate(
    hl.agg.count_where(hl.is_missing(key_ht.s))
) == 0, "Missing hashed IDs detected"
n_cols = vvar.count_cols()
n_distinct = var_keys.distinct().count()
assert n_cols == n_distinct, "Duplicate hashed sample IDs detected"

print("✓ All de-identification sanity checks passed")

# optional: peek at a few sample_info rows (safe, already hashed)
vvar.select_cols("sample_info").cols().show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

n cols (ref, var): 10 10
? All de-identification sanity checks passed
+--------------------------------------------------------------------+
| s                                                                  |
+--------------------------------------------------------------------+
| str                                                                |
+--------------------------------------------------------------------+
| "03757d3daaa258dd7c1f3a907622a6ce7ae20d4caf4d296071753e508832a27d" |
| "20d42e080ba16b8534a160a5cdf9e21da11f7bf568ffb25c7ce8636f2107c330" |
| "293a768734a3884ae92502b373799821bd2b745b26095b7a8ac5a05d2b0b4ecb" |
| "2d870aaee68611c396f7d470d2ed1a9522551812c4308738145348e9b9062c93" |
| "46fcf59185a71ab29ea6bcf7068a4c37005f32f572bba865fe42622fb8d0b02c" |
+--------------------------------------------------------------------+

+--------------------------------------------------------------------+
| sample_info.npmid                                                  |
+-----

In [16]:
# write output
# vds_deid.write(hashed_vds_uri, overwrite=True)
# print("Wrote:", hashed_vds_uri)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…